In [1]:
# Install tiktoken for BPE-based tokenization of text into subword units.
!pip3 install tiktoken

In [2]:
# Check the installed tiktoken version used for BPE tokenization.

import importlib
import tiktoken

print("tiktoken version:", importlib.metadata.version("tiktoken"))

tiktoken version: 0.13.0


In [3]:
# Load the GPT-2 tokenizer, which uses Byte Pair Encoding (BPE) to convert text into subword tokens for efficient processing by NLP models.
tokenizer = tiktoken.get_encoding("gpt2")

In [4]:
# Define a sample text string to be tokenized.
# The text contains normal words as well as a special token (<|endoftext|>).
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
    "of someunknownPlace."
)

# Convert the text into token IDs using the GPT-2 BPE tokenizer.
# The 'allowed_special' parameter tells the tokenizer to treat
# <|endoftext|> as a valid special token instead of ordinary text.
integers = tokenizer.encode(
    text,
    allowed_special={"<|endoftext|>"}
)

# Display the resulting list of integer token IDs.
# Each integer represents a token generated by the BPE tokenizer.
print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]


In [5]:
# Convert the list of token IDs back into the original text.
# The decode() method reconstructs the text by mapping each
# token ID to its corresponding token and combining them.
strings = tokenizer.decode(integers)

# Display the decoded text.
# The output should closely match the original input text,
# including the special token <|endoftext|>.
print(strings)

Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.


In [6]:
# Encode the text "Akwirw ier" into token IDs using the GPT-2 BPE tokenizer.
# Since the word is uncommon, BPE may split it into multiple subword tokens.
integers = tokenizer.encode("Akwirw ier")

# Display the generated token IDs.
# Each integer corresponds to a token or subword unit.
print(integers)

# Decode the token IDs back into text.
# This reconstructs the original string from the encoded tokens.
strings = tokenizer.decode(integers)

# Display the decoded text to verify that encoding and decoding are reversible.
print(strings)

[33901, 86, 343, 86, 220, 959]
Akwirw ier


**### CREATING INPUT-TARGET PAIRS**

In [7]:
# Open the file "the-verdict.txt" in read mode with UTF-8 encoding
with open("the-verdict.txt", "r", encoding="utf-8") as f:

    # Read the entire content of the file and store it in the variable raw_text
    raw_text = f.read()

# Convert the text into token IDs using the tokenizer
enc_text = tokenizer.encode(raw_text)

# Print the total number of tokens generated from the text
print(len(enc_text))

5145


In [8]:
# Create a new variable enc_sample that contains tokens starting from index 50 and continues until the end of enc_text
enc_sample = enc_text[50:]

In [9]:
# Define the context size (number of input tokens used to predict the next token)
context_size = 4

# The context_size of 4 means that the model looks at 4 consecutive tokens
# to learn and predict the next token in the sequence.
# Example:
# Input (x)  = [1, 2, 3, 4]
# Target (y) = [2, 3, 4, 5]
# Each target token is the next token corresponding to the input sequence.

# Select the first 4 tokens from enc_sample as the input sequence
x = enc_sample[:context_size]

# Select the next 4 tokens (shifted by one position) as the target sequence
y = enc_sample[1:context_size + 1]

# Print the input token sequence
print(f"x: {x}")

# Print the target token sequence
print(f"y:      {y}")

x: [290, 4920, 2241, 287]
y:      [4920, 2241, 287, 257]


In [10]:
# Loop through context lengths from 1 up to context_size (inclusive)
for i in range(1, context_size + 1):

    # Take the first i tokens as the input context
    context = enc_sample[:i]

    # Select the token immediately following the context as the target (desired output)
    desired = enc_sample[i]

    # Print the context and the corresponding target token
    print(context, "---->", desired)

[290] ----> 4920
[290, 4920] ----> 2241
[290, 4920, 2241] ----> 287
[290, 4920, 2241, 287] ----> 257


In [11]:
# Loop through context lengths from 1 up to context_size (inclusive)
for i in range(1, context_size + 1):

    # Take the first i tokens as the input context
    context = enc_sample[:i]

    # Select the token immediately following the context as the target token
    desired = enc_sample[i]

    # Convert the context tokens back into readable text
    # Convert the target token into a list before decoding because
    # tokenizer.decode() expects a sequence of token IDs
    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

 and ---->  established
 and established ---->  himself
 and established himself ---->  in
 and established himself in ---->  a


In [12]:
# Import Dataset and DataLoader classes from PyTorch
from torch.utils.data import Dataset, DataLoader


# Define a custom dataset class for GPT training
class GPTDatasetV1(Dataset):

    # Constructor method that initializes the dataset
    def __init__(self, txt, tokenizer, max_length, stride):

        # List to store input token sequences
        self.input_ids = []

        # List to store target token sequences
        self.target_ids = []

        # Convert the entire text into token IDs using the tokenizer
        # allowed_special allows special tokens such as <|endoftext|>
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        # Create overlapping input-target pairs using a sliding window
        # i moves from the beginning of the token list to the end
        # stride determines how many positions the window shifts each time
        for i in range(0, len(token_ids) - max_length, stride):

            # Extract a chunk of max_length tokens as the input sequence
            input_chunk = token_ids[i:i + max_length]

            # Extract the next chunk shifted by one position as the target sequence
            target_chunk = token_ids[i + 1: i + max_length + 1]

            # Convert the input chunk into a PyTorch tensor and store it
            self.input_ids.append(torch.tensor(input_chunk))

            # Convert the target chunk into a PyTorch tensor and store it
            self.target_ids.append(torch.tensor(target_chunk))

    # Returns the total number of training samples in the dataset
    def __len__(self):
        return len(self.input_ids)

    # Returns a specific input-target pair given an index
    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [13]:
# Define a function to create and return a DataLoader for GPT training
def create_dataloader_v1(txt, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    # Initialize the GPT-2 tokenizer for converting text into token IDs
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create a GPTDatasetV1 object that tokenizes the text and
    # generates input-target sequences using a sliding window
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create a DataLoader to efficiently load data in batches
    dataloader = DataLoader(
        dataset,

        # Number of samples (input-target pairs) per batch
        batch_size=batch_size,

        # Randomly shuffle the dataset at the beginning of each epoch
        shuffle=shuffle,

        # Drop the last batch if it contains fewer samples than batch_size
        drop_last=drop_last,

        # Number of worker processes used for loading data
        # 0 means data loading happens in the main process
        num_workers=num_workers
    )

    # Return the DataLoader object
    return dataloader

In [14]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

In [15]:
# Import the PyTorch library
import torch

# Display the installed PyTorch version
print("PyTorch version:", torch.__version__)

# Create a DataLoader from the raw text
# batch_size=1   -> Each batch contains one training example
# max_length=4   -> Each input sequence contains 4 tokens
# stride=1       -> Sliding window moves one token at a time
# shuffle=False  -> Data is loaded in sequential order

dataloader = create_dataloader_v1(
    raw_text,
    batch_size=1,
    max_length=4,
    stride=1,
    shuffle=False
)

# Create an iterator from the DataLoader
data_iter = iter(dataloader)

# Retrieve the first batch from the DataLoader
first_batch = next(data_iter)

# Print the first batch (input tokens and target tokens)
print(first_batch)

PyTorch version: 2.11.0+cpu
[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [16]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


In [17]:
# Create a DataLoader from the raw text
# batch_size=8  -> Each batch contains 8 input-target pairs
# max_length=4  -> Each input sequence contains 4 tokens
# stride=4      -> The sliding window moves 4 tokens at a time (non-overlapping chunks)
# shuffle=False -> Samples are loaded in the original order
dataloader = create_dataloader_v1(
    raw_text,
    batch_size=8,
    max_length=4,
    stride=4,
    shuffle=False
)

# Create an iterator from the DataLoader
data_iter = iter(dataloader)

# Retrieve the first batch of input and target sequences
inputs, targets = next(data_iter)

# Print the input token sequences
print("Inputs:\n", inputs)

# Print the corresponding target token sequences
print("\nTargets:\n", targets)

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])
